# Customer Churn Prediction Pipeline
**Author:** Mayank Joshi  
**Dataset:** Telco Customer Churn (Kaggle)  
**Goal:** Predict churn and generate a business retention recommendation.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

## 1. Load Data & SQL Exploration

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('../data/telco_churn_sample.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Load into SQLite for SQL-style exploration
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')

# Churn rate by contract type
query1 = '''
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
'''
print('Churn Rate by Contract Type:')
pd.read_sql(query1, conn)

In [ ]:
# Churn rate by tenure band
query2 = '''
SELECT
    CASE
        WHEN tenure <= 6 THEN '0–6 months'
        WHEN tenure <= 12 THEN '7–12 months'
        WHEN tenure <= 24 THEN '13–24 months'
        ELSE '2+ years'
    END AS tenure_band,
    COUNT(*) AS customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY tenure_band
ORDER BY churn_rate_pct DESC
'''
print('Churn Rate by Tenure Band:')
pd.read_sql(query2, conn)

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Binary target
df['churn_binary'] = (df['Churn'] == 'Yes').astype(int)

# Feature engineering
df['charge_per_month_ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
df['is_new_customer'] = (df['tenure'] <= 6).astype(int)
df['high_monthly_charge'] = (df['MonthlyCharges'] > 70).astype(int)

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', 'Churn']]

le = LabelEncoder()
for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

print(f'Features created. Dataset shape: {df.shape}')

## 3. Model Training

In [ ]:
feature_cols = (
    ['tenure', 'MonthlyCharges', 'TotalCharges',
     'charge_per_month_ratio', 'is_new_customer', 'high_monthly_charge'] +
    [c + '_enc' for c in cat_cols]
)

X = df[feature_cols]
y = df['churn_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:, 1])

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f'Logistic Regression — Accuracy: {lr.score(X_test_sc, y_test):.2%} | AUC: {lr_auc:.3f}')
print(f'Random Forest       — Accuracy: {rf.score(X_test, y_test):.2%} | AUC: {rf_auc:.3f}')

## 4. Evaluation Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- Confusion Matrix --
cm = confusion_matrix(y_test, rf.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Random Forest', fontweight='bold')

# -- ROC Curve --
for model, X_eval, label, color in [
    (lr, X_test_sc, 'Logistic Regression', '#42A5F5'),
    (rf, X_test, 'Random Forest', '#66BB6A')
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_eval)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_eval)[:, 1])
    axes[1].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison', fontweight='bold')
axes[1].legend()

# -- Feature Importance --
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top10 = importances.nlargest(10)
top10.sort_values().plot(kind='barh', ax=axes[2], color='#FFA726')
axes[2].set_title('Top 10 Feature Importances — Random Forest', fontweight='bold')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../visualisations/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Business Recommendation

In [ ]:
# High-risk segment: Month-to-month + Fibre Optic + High charges
df['churn_prob'] = rf.predict_proba(X)[:, 1]

high_risk = df[
    (df['Contract'] == 'Month-to-month') &
    (df['InternetService'] == 'Fiber optic') &
    (df['MonthlyCharges'] > 70)
]

avg_churn_prob = high_risk['churn_prob'].mean()
segment_size = len(high_risk)
avg_monthly_value = high_risk['MonthlyCharges'].mean()

print('=== BUSINESS RECOMMENDATION ===')
print(f'High-risk segment size:    {segment_size:,} customers ({100*segment_size/len(df):.1f}% of base)')
print(f'Average churn probability: {avg_churn_prob:.1%}')
print(f'Average monthly value:     £{avg_monthly_value:.2f}')
print()
print('Projected impact of proactive 12-month contract discount offer:')
retained = int(segment_size * avg_churn_prob * 0.30)  # 30% conversion assumed
annual_revenue_saved = retained * avg_monthly_value * 12
print(f'  Estimated customers retained:   {retained:,}')
print(f'  Estimated annual revenue saved: £{annual_revenue_saved:,.0f}')
print()
print('Recommendation: Prioritise outreach to Month-to-month Fibre Optic customers')
print('in months 1–6 of tenure with a discounted annual contract offer.')